вытащить в кармане не кармане в отдельный скрипт
добавить скрипт для поиска одинаковых структур с выбором лучшишь по ангстремам например плюс дикий тип 

X-ray, cryo-EM, model?
Holo state or apo state?
Revert mutations? Build missing residues? Cap termini? Protonate manually?

поправить лог

доавить файлы с удалёнными структурами

привести к дикому типу
добавить недостающее

доавить белки  Reduce (по умолчанию в DOCK3.7), Maestro (Schrödinger), PropKa или Chimera

низкая заполненность, высокие параметры смещения (B-значения), плохая электронная плотность

сплиты?\

сплиты с oddt

In [ ]:
import sys
from pathlib import Path

import numpy as np
from Bio.PDB import PDBParser
from loguru import logger

logger.remove()
logger.add(sys.stdout, format="{message}", level="INFO")

def calculate_distances(ligand_atoms, protein_atoms):
    """Calculate minimum distances between ligand atoms and protein atoms."""
    distances = []
    for lig_atom in ligand_atoms:
        lig_coord = lig_atom.get_coord()
        min_distance = np.min([np.linalg.norm(lig_coord - prot_atom.get_coord()) for prot_atom in protein_atoms])
        distances.append(min_distance)
    return distances

def is_ligand_buried(pdb_file, threshold, distance_cutoff=5.0):
    """
    Determine if the ligand in a PDB file is buried based on the fraction of ligand atoms
    within a cutoff distance from protein atoms.

    Parameters:
        pdb_file: Path to the PDB file.
        threshold: Fraction threshold to consider the ligand as buried.
        distance_cutoff: Distance cutoff in Ångströms.
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('complex', pdb_file)

    ligand_atoms = []
    protein_atoms = []

    for model in structure:
        for chain in model:
            for residue in chain:
                # Determine if the residue is a ligand or protein
                if residue.id[0].strip() == '' or residue.id[0].strip() == 'W':  # Protein residues
                    protein_atoms.extend(residue.get_atoms())
                else:  # Ligand
                    ligand_atoms.extend(residue.get_atoms())

    if not ligand_atoms or not protein_atoms:
        logger.info(f"Ligand or protein not found in {pdb_file.name}.")
        return False

    distances = calculate_distances(ligand_atoms, protein_atoms)

    buried_atoms = sum(1 for d in distances if d <= distance_cutoff)
    fraction_buried = buried_atoms / len(ligand_atoms)

    return fraction_buried >= threshold

def process_structures(output_dir, threshold=0.3, distance_cutoff=5.0):
    """
    Process the output directory to remove structures where the ligand is not buried.

    Parameters:
        output_dir: Directory containing the PDB files.
        threshold: Fraction threshold to consider the ligand as buried.
        distance_cutoff: Distance cutoff in Ångströms.
    """
    pdb_files = Path(output_dir).glob("*.pdb")
    for pdb_file in pdb_files:
        logger.info(f"Checking file {pdb_file.name}")

        if not is_ligand_buried(pdb_file, threshold, distance_cutoff):
            logger.info(f"Ligand in {pdb_file.name} is on the surface or not found, deleting file.")
            pdb_file.unlink()
        else:
            logger.info(f"Ligand in {pdb_file.name} is in a pocket, file retained.")


process_structures('separated_complexes', threshold=0.3, distance_cutoff=5.0)
